# Phase 1: QLoRA Fine-Tuning — Mistral-7B on Dolly-15K

**Runtime required:** GPU with ≥16GB VRAM
- Google Colab: Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro)
- Kaggle Notebooks: T4 x2 (free, 30hr/week)
- AWS: g4dn.xlarge (T4, ~$0.53/hr), g5.xlarge (A10G, ~$1.00/hr)

**What this notebook does:**
1. Installs dependencies
2. Loads Mistral-7B in 4-bit (QLoRA)
3. Attaches LoRA adapters
4. Fine-tunes on Dolly-15K instruction data
5. Saves adapter locally
6. Pushes adapter to HuggingFace Hub
7. Runs a quick inference test

## Step 0: Verify GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Also check with nvidia-smi
!nvidia-smi

## Step 1: Install Dependencies

**Why each package:**
- `transformers`: loads pretrained models and tokenizers
- `peft`: injects LoRA adapters
- `trl`: SFTTrainer for supervised fine-tuning
- `bitsandbytes`: enables 4-bit quantization (the Q in QLoRA)
- `accelerate`: handles device placement automatically
- `datasets`: loads HuggingFace datasets
- `huggingface_hub`: push/pull from HF Hub

In [ ]:
!pip install -q \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl==0.8.6" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.29.0" \
    "datasets>=2.18.0" \
    "huggingface_hub>=0.22.0" \
    sentencepiece \
    protobuf

# After install, restart the kernel so the newly installed versions are loaded:
# Kaggle: Run → Restart & Clear Output, then re-run from Step 0
# (Don't skip this — Kaggle pre-loads its own TRL version at kernel start)

## Step 2: Login to HuggingFace Hub

Get your token: https://huggingface.co/settings/tokens  
Create a token with **write** permissions (to push your adapter).

In [ ]:
from huggingface_hub import login
login()  # will prompt for your token

## Step 3: Load Model in 4-bit (QLoRA)

**What's happening:**
- `BitsAndBytesConfig(load_in_4bit=True)`: tells transformers to quantize weights to 4-bit
- `bnb_4bit_quant_type='nf4'`: NormalFloat4 — optimized for neural net weight distributions
- `bnb_4bit_use_double_quant=True`: quantize the quantization constants (saves ~0.4 bits/param)
- `device_map='auto'`: Accelerate places layers on GPU/CPU automatically

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model loaded. Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Step 4: Prepare Model for k-bit Training

`prepare_model_for_kbit_training` does 3 things:
1. Casts LayerNorm layers to float32 (stability during training)
2. Enables gradient checkpointing (recompute activations instead of storing them)
3. Enables input embeddings to receive gradients

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

## Step 5: Add LoRA Adapters

**Key parameters:**
- `r=16`: rank — expressiveness of the adapter. Higher = more capacity, more params
- `lora_alpha=32`: scaling (keep at 2×r by convention)
- `target_modules`: which weight matrices get adapters

After `get_peft_model`, check trainable parameters — should be ~0.5-2% of total.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)

# ALWAYS check this — confirms LoRA is applied correctly
model.print_trainable_parameters()
# Expected output: trainable params: ~20M || all params: ~3.75B || trainable%: ~0.53%

## Step 6: Load and Format Dataset

Dolly-15K sample structure:
```json
{"instruction": "...", "context": "...", "response": "...", "category": "..."}
```

We format each sample into Mistral's chat template:
```
<s>[INST] instruction \n\n context [/INST] response </s>
```

**Why response masking matters:**  
The model should only learn to predict the RESPONSE tokens.  
If we compute loss on instruction tokens too, the model learns to predict instructions,  
which is not what we want. SFTTrainer handles this automatically.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('databricks/databricks-dolly-15k', split='train')

# Use 5K samples for the foundation run (faster)
dataset = dataset.select(range(5000))

def format_sample(sample):
    instruction = sample['instruction']
    context = sample.get('context', '').strip()
    response = sample['response']

    user_msg = f"{instruction}\n\nContext: {context}" if context else instruction
    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': response},
    ]
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

dataset = dataset.map(format_sample, remove_columns=dataset.column_names)

print(f'Dataset: {len(dataset)} samples')
print('Sample:')
print(dataset[0]['text'][:400])

## Step 7: Train

**Key TrainingArguments explained:**
- `gradient_accumulation_steps=4`: accumulate gradients over 4 steps before updating
  → effective batch = 4 (per_device) × 4 (accumulation) = 16
- `optim='paged_adamw_32bit'`: Adam optimizer with paged memory (prevents OOM)
- `lr_scheduler_type='cosine'`: learning rate follows cosine curve (smooth decay)
- `warmup_ratio=0.03`: first 3% of steps, lr linearly increases to target value

**Expected training time:**
- T4 GPU (Colab free): ~45-60 min for 5K samples, 1 epoch
- A10G (Colab Pro / AWS g5): ~20-30 min

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./outputs/phase1-mistral-dolly',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    optim='paged_adamw_32bit',
    gradient_checkpointing=True,
    group_by_length=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    args=training_args,
    packing=False,
)

trainer.train()

## Step 8: Save Adapter

In [ ]:
ADAPTER_PATH = './outputs/phase1-mistral-dolly/final-adapter'

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

import os
files = os.listdir(ADAPTER_PATH)
print(f'Adapter saved. Files: {files}')
# Expected: adapter_config.json, adapter_model.safetensors, tokenizer files

## Step 9: Push Adapter to HuggingFace Hub

Change `YOUR_HF_USERNAME` to your actual HuggingFace username.

In [ ]:
HF_USERNAME = 'YOUR_HF_USERNAME'  # <-- change this
REPO_ID = f'{HF_USERNAME}/mistral-7b-dolly-qlora'

trainer.model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

print(f'Adapter at: https://huggingface.co/{REPO_ID}')

## Step 10: Quick Inference Test

Test the fine-tuned model before proceeding to Phase 2.

In [ ]:
model.eval()

def generate(instruction, context='', max_new_tokens=256):
    user_msg = f"{instruction}\n\nContext: {context}" if context else instruction
    messages = [{'role': 'user', 'content': user_msg}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# Test 1: Open QA
print('=== Test 1: Open QA ===')
print(generate('What are the main causes of climate change?'))

print('\n=== Test 2: Summarization ===')
print(generate(
    'Summarize the following in 2 sentences.',
    context='The Amazon rainforest is the world\'s largest tropical rainforest, covering over 5.5 million square kilometers. It is home to an estimated 10% of all species on Earth and plays a crucial role in regulating the global climate by absorbing vast amounts of CO2.'
))

print('\n=== Test 3: Brainstorming ===')
print(generate('Give me 5 creative names for a coffee shop that focuses on sustainability.'))

## Next Steps

Phase 1 complete. You now have a working QLoRA fine-tuning pipeline.

**Phase 2 domains:**
- `02_medical.ipynb` — MedQA, PubMedQA datasets
- `03_legal.ipynb` — LegalBench, contract analysis datasets  
- `04_finance.ipynb` — FinQA, earnings call datasets
- `05_coding.ipynb` — CodeAlpaca, CodeInstructions datasets

Each domain notebook follows the same structure but with:
- A different dataset and formatting function
- Adjusted `max_seq_length` (legal/finance need 1024-2048)
- Domain-specific evaluation prompts